In [6]:
from cmapss.data import load_cmapss_data, load_rul_labels, SENSOR_COLS, FD001_DEAD_SENSORS
from cmapss.features import (
    add_rul_column, split_units, fit_scaler, apply_scaler,
    create_sequences, create_test_sequences,
)

TRAIN_PATH = "../data/raw/train_FD001.txt"
TEST_PATH = "../data/raw/test_FD001.txt"
RUL_PATH = "../data/raw/RUL_FD001.txt"

RUL_CAP = 125       # 커뮤니티 표준값 (120~130 범위), 최소 수명 128과도 스케일이 맞음
WINDOW_SIZE = 30
VAL_RATIO = 0.2
SEED = 42

In [12]:
df = load_cmapss_data(TRAIN_PATH)
feature_cols = [i for i in SENSOR_COLS if i not in FD001_DEAD_SENSORS]
print(f"N of available features {len(feature_cols)}:", feature_cols)

# cap에 걸린 row 비율 극단적으로 높거나 낮으면 cap 재조정 신호
df = add_rul_column(df, rul_cap=RUL_CAP)
cap_ratio = (df["RUL"] == RUL_CAP).mean()
df[["unit", "cycle", "RUL"]].head()

N of available features 14: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


,unit,cycle,RUL
0,1,1,125
1,1,2,125
2,1,3,125
3,1,4,125
4,1,5,125


In [8]:
# Train/Val Split — unit(엔진) 단위
tr_df, val_df, tr_units, val_units = split_units(df, val_ratio=VAL_RATIO, seed=SEED)
overlap = set(tr_units) & set(val_units)

# Normalize
scaler = fit_scaler(tr_df, feature_cols)
tr_scaled = apply_scaler(tr_df, scaler, feature_cols)
val_scaled = apply_scaler(val_df, scaler, feature_cols)

# 슬라이딩 윈도우 생성 (train/val — 엔진당 여러 윈도우)
X_train, y_train, _ = create_sequences(tr_scaled, feature_cols, WINDOW_SIZE)
X_val, y_val, _ = create_sequences(val_scaled, feature_cols, WINDOW_SIZE)
print("X_train", X_train.shape, "X_val", X_val.shape)

X_train (14241, 30, 14) X_val (3490, 30, 14)


In [9]:
test_df = load_cmapss_data(TEST_PATH)
test_scaled = apply_scaler(test_df, scaler, feature_cols)
X_test, test_units = create_test_sequences(test_scaled, feature_cols, WINDOW_SIZE)
y_test = load_rul_labels(RUL_PATH)

print("X_test", X_test.shape, "N of the answers", len(y_test))
assert X_test.shape[0] == len(y_test)
print("Validation Successful: equalivant number of engines and answers")
assert X_val.shape[0] == len(y_val)

X_test (100, 30, 14) N of the answers 100
Validation Successful: equalivant number of engines and answers


In [10]:
# Save dataset
#`data/interim/`: RUL 라벨까지 붙인 중간 df (아직 배열은 아님)
#`data/processed/`: 모델에 바로 넣을 수 있는 최종 배열

import os
import numpy as np

os.makedirs("../data/interim", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

tr_df.to_parquet("../data/interim/train_with_rul.parquet")
val_df.to_parquet("../data/interim/val_with_rul.parquet")

np.save("../data/processed/X_train.npy", X_train)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/X_val.npy", X_val)
np.save("../data/processed/y_val.npy", y_val)
np.save("../data/processed/X_test.npy", X_test)
np.save("../data/processed/y_test.npy", y_test)
print("Saved at data/interim, data/processed")

Saved at data/interim, data/processed
